# Log Analysis — Processing Durations

Parses logs, extracts per-frame processing durations, builds a summary DataFrame, plots distributions, and highlights outliers.

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

## Extract data

In [ ]:
# parse file into dict
pattern = re.compile(r"DEBUG processing: \(([0-9a-f]{6})\) (.+?) duration: ([\d.]+) ms")

STEP_COLS = {
    "Hash": "hash_ms",
    "Motion detection": "motion_ms",
    "Object detection": "object_ms",
    "Processing": "total_ms",
}

raw: dict[str, dict] = defaultdict(dict)

with Path("../20260406_174406_logs.txt").open() as f:
    for line in f:
        m = pattern.search(line)
        if not m:
            continue
        frame_hash, step, value = m.group(1), m.group(2), float(m.group(3))
        col = STEP_COLS.get(step, step.lower().replace(" ", "_") + "_ms")
        raw[frame_hash][col] = value

In [ ]:
# convert to dataframe
df = pd.DataFrame.from_dict(raw, orient="index")
df.index.name = "hash"
df["other_ms"] = (
    df["total_ms"] - df.drop(columns="total_ms").fillna(0).sum(axis=1)
).round(2)
df.describe()

## Duration evolution

In [ ]:
normalised = df.reset_index(drop=True) / df.median()

fig, ax = plt.subplots()

normalised.plot(logy=True, ax=ax)
ax.set_ylabel("× median")
ax.set_xlabel("Frame index")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

## Distribution of Processing Durations

In [ ]:
_labels = {v: k for k, v in STEP_COLS.items()}
DURATION_COLS = {
    col: _labels.get(col, col.removesuffix("_ms").replace("_", " ").title())
    for col in df.columns
}

fig, axes = plt.subplots(1, len(DURATION_COLS), figsize=(18, 4))
fig.suptitle("Distribution of Processing Durations (ms)")

for ax, (col, label) in zip(axes, DURATION_COLS.items()):
    data = df[col].dropna()
    ax.hist(data, bins=30)
    ax.set_title(label)
    ax.set_xlabel("Duration (ms)")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Outlier Detection

In [ ]:
def iqr_upper_bound(series: pd.Series) -> float:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return q3 + 1.5 * (q3 - q1)


records = []
for col, label in DURATION_COLS.items():
    if label in ["Other", "Processing"]:
        continue
    data = df[col].dropna()
    hi = iqr_upper_bound(data)
    outliers = df[col][df[col] > hi]
    for hash_val, val in outliers.items():
        records.append(
            {
                "hash": hash_val,
                "column": label,
                "value_ms": val,
                "upper_bound": round(hi, 2),
                "multiple": round(val / hi, 2),
            }
        )

df_out = pd.DataFrame(records)
df_out.sort_values("multiple", ascending=False)

## Frame rate analysis

In [ ]:
# quantile frame rate possibilities
df_with_obj = df.loc[df["object_ms"].notna()]

print(f"Processing duration for {len(df_with_obj)} frames with object detection: (ms):")
print(
    f"  p80    {df_with_obj["total_ms"].quantile(0.80):.1f}  →  {1000/df_with_obj["total_ms"].quantile(0.80):.1f} fps"
)
print(
    f"  p90    {df_with_obj["total_ms"].quantile(0.90):.1f}  →  {1000/df_with_obj["total_ms"].quantile(0.90):.1f} fps"
)
print(
    f"  p95    {df_with_obj["total_ms"].quantile(0.95):.1f}  →  {1000/df_with_obj["total_ms"].quantile(0.95):.1f} fps"
)
print(
    f"  p99    {df_with_obj["total_ms"].quantile(0.99):.1f}  →  {1000/df_with_obj["total_ms"].quantile(0.99):.1f} fps"
)
print(
    f"  max    {df_with_obj["total_ms"].max():.1f}  →  {1000/df_with_obj["total_ms"].max():.1f} fps"
)

In [ ]:
# breakdown of frame driving processes
means = df_with_obj.mean()
medians = df_with_obj.median()

summary = pd.DataFrame(
    {
        "mean_ms": means.drop(columns="total_ms"),
        "median_ms": medians.drop(columns="total_ms"),
        "mean_%": (means.drop(columns="total_ms") / means["total_ms"] * 100),
    }
).rename(index=DURATION_COLS)
summary.round(2)